In [1]:
cd /content/drive/MyDrive/김원/Colab_Notebooks

/content/drive/MyDrive/김원/Colab_Notebooks


In [2]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 14.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.6.1
    Uninstalling fsspec-2024.6.1:
      Successfully uninstalled fsspec-2024.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.3.1+cu121 requ

In [3]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00


In [4]:
import torch
from datasets import load_dataset
from transformers import(
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import pandas as pd
import numpy as np
import evaluate

In [5]:
dataset = load_dataset('sepidmnorozy/Korean_sentiment')
dataset

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/36000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1333 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2667 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 36000
    })
    validation: Dataset({
        features: ['label', 'text'],
        num_rows: 1333
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 2667
    })
})

In [6]:
print(dataset['train'][3118])
print(dataset['train'][14310])

{'label': 1, 'text': '졸잼!!!성아가나중에억울한일이잇어서좀슬펏는데마지막은기쁘게끝나서다행이에여'}
{'label': 0, 'text': '진짜 어떻게 된놈의 영화가 여고괴담 1보다도 못함? 신기하다 그것도 2012년작이 1998년보다 못함 솔까 여고괴담1은 반전은 최고지 뭐 이놈의 영화는 여고괴담 시리즈보다도 못하는거같다'}


In [7]:
model_name = 'kykim/bert-kor-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer

tokenizer_config.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/344k [00:00<?, ?B/s]

BertTokenizerFast(name_or_path='kykim/bert-kor-base', vocab_size=42000, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [8]:
def tokenizer_func(x):
    return tokenizer(
        x['text'],
        padding='max_length',
        max_length=256,
        truncation=True
    )

In [9]:
tokenized_datasets = dataset.map(tokenizer_func, batched=True)

Map:   0%|          | 0/36000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1333 [00:00<?, ? examples/s]

Map:   0%|          | 0/2667 [00:00<?, ? examples/s]

In [10]:
train_num_samples = 10000

train_ds = tokenized_datasets['train'].shuffle(seed=42).select(range(train_num_samples))
eval_ds = tokenized_datasets['validation'].shuffle(seed=42)

# 전이학습 Transfer Learning

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

pytorch_model.bin:   0%|          | 0.00/476M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at kykim/bert-kor-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# HyperParams

In [12]:
bs = 32
epochs = 4
lr = 1e-5

In [13]:
args = TrainingArguments(
    'outputs',
    learning_rate = lr,
    warmup_ratio = 0.1,
    lr_scheduler_type = 'cosine',
    fp16 = True,
    evaluation_strategy='epoch',
    per_device_train_batch_size = bs,
    per_device_eval_batch_size = bs,
    gradient_accumulation_steps=4, #until bs =128
    eval_accumulation_steps = 4,
    num_train_epochs=epochs,
    weight_decay=0.01,
    report_to = 'none'
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


# Metrics

In [14]:
metric = evaluate.load('accuracy')

# all Transformers models retrun logits
def compute_metrics(eval_pred):
    logits, labels =eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

In [15]:
class CustomTrainer(Trainer):
    def _save_checkpoint(self, model, trial, metrics=None):
        # 모델을 저장하기 전에 모든 텐서를 contiguous로 만듦
        for name, param in model.named_parameters():
            if param is not None:
                param.data = param.data.contiguous()
                if param.grad is not None:
                    param.grad.data = param.grad.data.contiguous()
        super()._save_checkpoint(model, trial, metrics)

# Trainer

In [16]:
trainer = CustomTrainer(
    model,
    args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
0,No log,0.337200,0.873218
1,No log,0.306579,0.876219
2,No log,0.295478,0.881470
3,No log,0.294481,0.884471


TrainOutput(global_step=312, training_loss=0.34380868765024036, metrics={'train_runtime': 149.2031, 'train_samples_per_second': 268.091, 'train_steps_per_second': 2.091, 'total_flos': 5247486888099840.0, 'train_loss': 0.34380868765024036, 'epoch': 3.987220447284345})

In [ ]:
trainer.save_model('./preprocess_transformer_model')

In [18]:
pipe = pipeline('text-classification', model='./preprocess_transformer_model')

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


# 테스트셋 사용

In [ ]:
test_data = dataset['validation'].shuffle(seed=424)[:100]
td = pd.DataFrame(test_data)
td

,label,text
0,1,"비밀은 적정선에 관한 이야기이기도 하다. 사람에게 돈이란, 평판이란, 책임이란, 기..."
1,0,메인 시나리오 뼈대가 없고 보여주기라기엔 용과의 전투씬도 너무 부족함..더욱이 결말...
2,1,올??? 찌질한 스커드가 노만 리더스 행님이었다니 !! ㅋㅋ
3,0,5점이 적당하나 평점이 쓸데없이 높아서 낮출 필요가 있슴...
4,0,"연출, 기획, 스토리 전부다 쓰레기..제인생 영화중에 제일 쓰레기.. 절대보지마요진심"
...,...,...
95,1,"""""""철아 엄마가 꼭 니 신세 갚고 죽을께"""""""
96,0,이건 뭐 .. 완전 삼류네
97,0,으... 밑도 끝도 없음. 여자가 처음부터 자기혈청을 제공했으면 카오스가 안왔을거 아님?
98,0,요즘에 보기에는 상당히 지루한 진부한 스릴러. 원초적 본능을 뛰어 넘기에도 힘든 영...


In [ ]:
preds = pipe(td['text'].tolist())

preds_df = pd.DataFrame(preds)
preds_df

,label,score
0,LABEL_1,0.977509
1,LABEL_0,0.987574
2,LABEL_0,0.929627
3,LABEL_1,0.638549
4,LABEL_0,0.987248
...,...,...
95,LABEL_1,0.720594
96,LABEL_0,0.985785
97,LABEL_0,0.961799
98,LABEL_0,0.987525


In [ ]:
preds_df.rename(columns={'label':'pred'}, inplace=True)
preds_df['pred'] = preds_df['pred'].map({'LABEL_1': 1, 'LABEL_0':0})

preds_df = pd.concat([preds_df, td], axis=1)
preds_df

,pred,score,label,text
0,1,0.977509,1,"비밀은 적정선에 관한 이야기이기도 하다. 사람에게 돈이란, 평판이란, 책임이란, 기..."
1,0,0.987574,0,메인 시나리오 뼈대가 없고 보여주기라기엔 용과의 전투씬도 너무 부족함..더욱이 결말...
2,0,0.929627,1,올??? 찌질한 스커드가 노만 리더스 행님이었다니 !! ㅋㅋ
3,1,0.638549,0,5점이 적당하나 평점이 쓸데없이 높아서 낮출 필요가 있슴...
4,0,0.987248,0,"연출, 기획, 스토리 전부다 쓰레기..제인생 영화중에 제일 쓰레기.. 절대보지마요진심"
...,...,...,...,...
95,1,0.720594,1,"""""""철아 엄마가 꼭 니 신세 갚고 죽을께"""""""
96,0,0.985785,0,이건 뭐 .. 완전 삼류네
97,0,0.961799,0,으... 밑도 끝도 없음. 여자가 처음부터 자기혈청을 제공했으면 카오스가 안왔을거 아님?
98,0,0.987525,0,요즘에 보기에는 상당히 지루한 진부한 스릴러. 원초적 본능을 뛰어 넘기에도 힘든 영...


In [ ]:
mask = preds_df['pred'] == preds_df['label']
len(preds_df[mask])

88

# 긍/부정 예측이 필요한 데이터셋

In [19]:
file_path = ['/content/drive/MyDrive/김원/Datas/Cream_dataset_v1.csv',
             '/content/drive/MyDrive/김원/Datas/Essence_dataset_v1.csv',
             '/content/drive/MyDrive/김원/Datas/Lotion_dataset_v1.csv',
             '/content/drive/MyDrive/김원/Datas/Mist_dataset_v1.csv',
             '/content/drive/MyDrive/김원/Datas/Skin_dataset_v1.csv']
dataframes = []

for file in file_path:
    df = pd.read_csv(file)
    dataframes.append(df)

raw = pd.concat(dataframes, ignore_index=True)
df = raw.copy()

In [20]:
df.head()

,Unnamed: 0,name,brand,price,sale_price,picture,url,volume,skin_type,ingredient,review
0,0.0,ahc유스래스팅리얼아이크림포페이스스페셜링클케어,AHC,23000,16900.0,https://image.oliveyoung.co.kr/uploads/images/...,https://www.oliveyoung.co.kr/store/G.do?goodsN...,본품: 아이크림 35ml / 증정품: 본품 동일 아이크림 7ml * 1ea,모든 피부용,"포트마리골드단백질추출물, 락토바실러스발효용해물, 글리세린, 다이프로필렌글라이콜, 사...",['세안 혹은 샤워 후 얼굴에 전체적으로 바르고 있습니다. 자극 없이 순해서 좋어요...
1,1.0,아비브수분초히알루론크림하이드레이팅팟,아비브,35000,35000.0,https://image.oliveyoung.co.kr/uploads/images/...,https://www.oliveyoung.co.kr/store/G.do?goodsN...,80ml (+수분초 패드 25ml/10pads 증정),모든 피부용,"정제수, 돌나물추출물, 글리세린, 세틸에틸헥사노에이트, 카프릴릭/카프릭트라이글리세라...","['ㅜㅜ강추.. 재발 사보셔요!!! 수분크림 유목민 탈출했엉요!!', '사용하면 피..."
2,2.0,ootdpm오버나잇아이크림,OOTD,30000,30000.0,https://image.oliveyoung.co.kr/uploads/images/...,https://www.oliveyoung.co.kr/store/G.do?goodsN...,25G,모든 피부,"정제수, 부틸렌글라이콜, 글리세린, 카프릴릭/카프릭트라이글리세라이드, 해바라기씨오일...","['지성 복합성에 너무좋아요 있던 주름도 사라집니다\\n향도 좋아요', '자기전에 ..."
3,3.0,ootdam인텐스아이세럼,OOTD,33000,17900.0,https://image.oliveyoung.co.kr/uploads/images/...,https://www.oliveyoung.co.kr/store/G.do?goodsN...,30g,모든 피부,"정제수, 글리세린, 프로필렌글라이콜, 세틸에틸헥사노에이트, 하이드로제네이티드폴리데센...",['너무나 잘 쓰고 있어요!\\n발림성도 너무나 좋고 바르고 나서도 가벼운 느낌이라...
4,4.0,dr다룸베리어크림,다룸,28000,19600.0,https://image.oliveyoung.co.kr/uploads/images/...,https://www.oliveyoung.co.kr/store/G.do?goodsN...,75g,모든 피부용,정제수@글리세린@카프릴릴메치콘@디프로필렌글라이콜@이소아밀라우레이트@펜틸렌글라이콜@1...,['코시국이라 마스크를 쓰고 다녀서 그런지 얼굴 피부가 예민해져서 붉어지고 좁쌀여드...


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 778 entries, 0 to 777
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  512 non-null    float64
 1   name        778 non-null    object 
 2   brand       778 non-null    object 
 3   price       778 non-null    object 
 4   sale_price  778 non-null    object 
 5   picture     778 non-null    object 
 6   url         778 non-null    object 
 7   volume      778 non-null    object 
 8   skin_type   778 non-null    object 
 9   ingredient  778 non-null    object 
 10  review      778 non-null    object 
dtypes: float64(1), object(10)
memory usage: 67.0+ KB


In [21]:
df['review_list'] = df['review'].apply(lambda x: x.split(','))

In [22]:
df['review_list'].head()

,review_list
0,[['세안 혹은 샤워 후 얼굴에 전체적으로 바르고 있습니다. 자극 없이 순해서 좋어...
1,"[['ㅜㅜ강추.. 재발 사보셔요!!! 수분크림 유목민 탈출했엉요!!', '사용하면..."
2,"[['지성 복합성에 너무좋아요 있던 주름도 사라집니다\\n향도 좋아요', '자기전..."
3,[['너무나 잘 쓰고 있어요!\\n발림성도 너무나 좋고 바르고 나서도 가벼운 느낌이...
4,[['코시국이라 마스크를 쓰고 다녀서 그런지 얼굴 피부가 예민해져서 붉어지고 좁쌀여...


In [23]:
len(df['review_list'])

778

In [30]:
# 중복 제거 및 Null 값 제거
df.drop_duplicates(subset=['review_list'], inplace=True)
df.dropna(subset=['review_list'], inplace=True)

In [25]:
df.drop(columns='Unnamed: 0', inplace = True)

In [26]:
df['review_list'][0]

["['세안 혹은 샤워 후 얼굴에 전체적으로 바르고 있습니다. 자극 없이 순해서 좋어요'",
 " '한달 사용 리뷰입니다. 추천드립니다. 좋은상품입니다.'",
 " '매달 살 정도로 합리적인 가격과 좋은 제품을 만나게 해주셔서 감사하고 꾸준한 품질관리 해주는만큼 믿고 사게 되는 것 같아요'",
 " '눈가에 주름이 생기는거 같아 아이크링 유명한걸로\\\\n또세일하길래 구매했어요\\\\n효과가 좋았으면해요'",
 " '처음 써보는 건데 아직 특별한 점은 잘 못 느끼겠어요 좀 더 써봐야 할 듯해요'",
 " '발림성 좋고 끈적임이 없어서 무난하게 사용하기 좋은 거 같아요!'",
 " '이제품 몇년간 진짜 꾸준히 사용중인데 좋아요 눈가 미간 팔짜주름 목주름등등 저는 전체적으로 다 펴 바르는데\\\\n너무 촉촉하고 좋아요 끈적임도 많이 없고 수분감도 있어서 쓰기\\\\n딱 좋습니다\\\\n발림성도 촉촛해서 너무 좋고 눈가에 자극 또한 없어서 쓰기 편해요'",
 " '동생이 써보고싶다고 사봤는데 역시 좋은 거 같아요~ 부드럽고 좋네용'",
 " '자극없이 순하고 촉촉해서 믿고 사는 제품입니다!\\\\n자극없이 순하고 촉촉해서 믿고 사는 제품입니다!'",
 " '여기 브랜드 제품의 아이크림은 꾸준히 사용 중이에요 촉촉하고 무겁지 않아서 너무 마음에 들어요'",
 " '조금씩 사용해서 그런지 큰 효과는 안나타나서 더 꾸준히 사용해 보겠습니다.'",
 " '보습도 좋고 향도 좋고 무엇보다 순해서 좋은 것 같야요 엄마 선뮬 해드렸는데 좋아하셨습다'",
 " '무난하게 사용하기 좋은 아이크림 입니다! 주름 개선도 좋고 끈적거리지 않아서 좋네요!'",
 " '아이크림으로ㅠ유명한 에이에이치씨 이번에 원플러스 원으로 잘 샀습ㅋ'",
 " '외국에 사는 친구 선물로 구매했어요 (｡･･｡) ..! 좋습니다'",
 " '오랫만에 구매해봤어요. 아직 사용전이긴한데 예전 빨강색이였을때에도 자극없이 괜찮앗아서 다시 한번 써보려구요.'",
 " '촉촉하고 페이스 겸용되니까 편함!!!\\\\n끈적

In [27]:
# 텍스트 정규화 함수
def text_normalization(text):
    text = text.lower()  # 소문자 변환
    text = re.sub(r'\d+', '', text)  # 숫자 제거
    text = re.sub(r'\s+', ' ', text)  # 여러 개의 공백을 하나의 공백으로 치환
    text = re.sub(r'[^\w\s]', '', text)  # 특수 문자 제거
    return text

In [28]:
# review_list의 각 요소에 대해 text_normalization 적용
import re
df['review_list'] = df['review_list'].apply(lambda reviews: [text_normalization(review) for review in reviews])

In [32]:
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.6/488.6 kB 36.5 MB/s eta 0:00:00


In [33]:
# 불용어 제거 함수
from konlpy.tag import Okt
def remove_stopwords(text, stopwords):
    okt = Okt()
    words = okt.morphs(text)
    words = [word for word in words if word not in stopwords]
    return ' '.join(words)

In [34]:
# 불용어 리스트
stopwords = ['이', '그', '저', '것', '수', '들', '등', '을', '를', '은', '는', '에', '의', '가', '로', '에서']

In [35]:
# review_list의 각 요소에 대해 불용어 제거 적용
df['review_list'] = df['review_list'].apply(lambda reviews: [remove_stopwords(review, stopwords) for review in reviews])

In [37]:
# 모든 리뷰 리스트를 하나의 리스트로 축소
all_reviews = [review for sublist in df['review_list'] for review in sublist]

In [41]:
len(all_reviews)

79853

In [42]:
# 새로운 데이터프레임 생성
df_review_list = pd.DataFrame(all_reviews, columns=['review_list'])

In [43]:
df_review_list.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79853 entries, 0 to 79852
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_list  79853 non-null  object
dtypes: object(1)
memory usage: 624.0+ KB


In [44]:
df_review_list.to_csv('/content/drive/MyDrive/김원/Datas/preprecessed_total_reviews.csv')

In [ ]:
df_review_list = pd.read_csv('/content/drive/MyDrive/김원/Datas/preprecessed_total_reviews.csv')


In [45]:
# 데이터타입 변환
df_review_list['review_list'] = df_review_list['review_list'].astype(str)

In [46]:
len(df_review_list['review_list'])

79853

In [ ]:
df_review_list['review_list']

0             세안 혹은 샤워 후 얼굴 전체 적 으로 바르고 있습니다 자극 없이 순해서 좋어요
1                          한 달 사용 리뷰 입니다 추천 드립니다 좋은 상품 입니다
2        매달 살 정도 합리 적 인 가격 과 좋은 제품 만나게 해주셔서 감사하고 꾸준한 품질...
3        눈가 주름 생기는거 같아 아이크 링 유명한 걸 n 또세 일하길래 구매 했어요 n 효...
4            처음 써 보는 건데 아직 특별한 점 잘 못 느끼겠어요 좀 더 써 봐야 할 듯 해요
                               ...                        
79848    n 각질 에도 좋고 일단 순한 성분 진정 되는 느낌 좋아서 매일 쓰고 있습니다 n ...
79849                          재구매 했어용 피부 편안한 느낌 적 인 느낌 ㅎㅎ
79850    기초 케어 라인 전부 후시 다인 으로 통일 해서 쓰고 있는데 유수 분 밸런스 조절 ...
79851    후시 다인 제품 n 사용 해보고 반해가지고 n 요즘 매 일 매일 사용 하다 보 니깐...
79852    물 토너 인데 도 불구 하고 발림 성 뻐덕뻐덕해 요 n 자기 전 스킨 케어 썼더니 ...
Name: review_list, Length: 79853, dtype: object

In [47]:
# 데이터프레임을 데이터셋으로 변환
from datasets import Dataset

dataset = Dataset.from_pandas(df_review_list)

In [48]:
dataset

Dataset({
    features: ['review_list'],
    num_rows: 79853
})

In [49]:
# 모델 이름과 토크나이저 초기화
model_name = 'kykim/bert-kor-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [50]:
# 토크나이저 함수 정의
def tokenizer_func(examples):
    return tokenizer(
        examples['review_list'],
        padding='max_length',
        max_length=256,
        truncation=True
    )

In [51]:
tokenized_review_list = dataset.map(tokenizer_func, batched=True)

Map:   0%|          | 0/79853 [00:00<?, ? examples/s]

In [52]:
# 데이터셋 크기 확인
dataset_size = len(tokenized_review_list)
train_num_samples = min(79853, dataset_size)  # 데이터셋 크기를 넘지 않도록 설정

In [53]:
txts_td = tokenized_review_list.select(range(train_num_samples))

In [54]:
txts_td

Dataset({
    features: ['review_list', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 79853
})

In [55]:
preds_txts = pipe(txts_td['review_list'])

In [56]:
preds_txts_df = pd.DataFrame(preds_txts)

In [57]:
preds_txts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79853 entries, 0 to 79852
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   label   79853 non-null  object 
 1   score   79853 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.2+ MB


In [ ]:
preds_txts_df.head()

,label,score
0,1,0.978707
1,1,0.961900
2,1,0.988946
3,1,0.900858
4,1,0.547701


In [58]:
preds_txts_df['label'] = preds_txts_df['label'].map({'LABEL_1':1, 'LABEL_0':0})

In [59]:
review_labeling_df = pd.DataFrame(txts_td)

In [60]:
review_labeling_df.head()

,review_list,input_ids,token_type_ids,attention_mask
0,세안 혹은 샤워 후 얼굴 전체 적 으로 바르고 있습니다 자극 없이 순해서 좋어요,"[2, 17951, 15427, 15767, 7876, 14369, 14433, 6...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,한 달 사용 리뷰 입니다 추천 드립니다 좋은 상품 입니다,"[2, 7653, 3118, 13978, 15282, 14295, 14195, 18...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ..."
2,매달 살 정도 합리 적 인 가격 과 좋은 제품 만나게 해주셔서 감사하고 꾸준한 품질...,"[2, 18925, 4890, 14061, 17530, 6016, 5925, 140...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,눈가 주름 생기는거 같아 아이크 링 유명한 걸 n 또세 일하길래 구매 했어요 n 효...,"[2, 31115, 18622, 18644, 8132, 15565, 13999, 8...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,처음 써 보는 건데 아직 특별한 점 잘 못 느끼겠어요 좀 더 써 봐야 할 듯 해요,"[2, 14121, 5301, 15024, 20217, 14189, 16250, 6...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [61]:
preds_txts_df = pd.concat([preds_txts_df, review_labeling_df], axis=1)
preds_txts_df

,label,score,review_list,input_ids,token_type_ids,attention_mask
0,1,0.978707,세안 혹은 샤워 후 얼굴 전체 적 으로 바르고 있습니다 자극 없이 순해서 좋어요,"[2, 17951, 15427, 15767, 7876, 14369, 14433, 6...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,1,0.961900,한 달 사용 리뷰 입니다 추천 드립니다 좋은 상품 입니다,"[2, 7653, 3118, 13978, 15282, 14295, 14195, 18...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ..."
2,1,0.988946,매달 살 정도 합리 적 인 가격 과 좋은 제품 만나게 해주셔서 감사하고 꾸준한 품질...,"[2, 18925, 4890, 14061, 17530, 6016, 5925, 140...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,1,0.900858,눈가 주름 생기는거 같아 아이크 링 유명한 걸 n 또세 일하길래 구매 했어요 n 효...,"[2, 31115, 18622, 18644, 8132, 15565, 13999, 8...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,1,0.547701,처음 써 보는 건데 아직 특별한 점 잘 못 느끼겠어요 좀 더 써 봐야 할 듯 해요,"[2, 14121, 5301, 15024, 20217, 14189, 16250, 6...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
...,...,...,...,...,...,...
79848,1,0.952779,n 각질 에도 좋고 일단 순한 성분 진정 되는 느낌 좋아서 매일 쓰고 있습니다 n ...,"[2, 2054, 16948, 32135, 14410, 14405, 20212, 1...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
79849,1,0.984017,재구매 했어용 피부 편안한 느낌 적 인 느낌 ㅎㅎ,"[2, 15307, 36934, 8114, 14081, 17467, 14066, 6...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ..."
79850,1,0.964095,기초 케어 라인 전부 후시 다인 으로 통일 해서 쓰고 있는데 유수 분 밸런스 조절 ...,"[2, 16144, 16862, 15953, 16047, 7876, 8118, 31...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
79851,1,0.978629,후시 다인 제품 n 사용 해보고 반해가지고 n 요즘 매 일 매일 사용 하다 보 니깐...,"[2, 7876, 8118, 3110, 8159, 13996, 2054, 13978...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [62]:
preds_txts_df.drop(columns=['input_ids', 'token_type_ids', 'attention_mask'], inplace=True)

In [63]:
preds_txts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79853 entries, 0 to 79852
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   label        79853 non-null  int64  
 1   score        79853 non-null  float64
 2   review_list  79853 non-null  object 
dtypes: float64(1), int64(1), object(1)
memory usage: 1.8+ MB


In [64]:
preds_txts_df.to_csv('/content/drive/MyDrive/김원/Datas/total_review_labeling_without_clssification.csv')

In [65]:
# 같은 행에서 df['label']의 값이 1이면 df['review_list']에 [긍정]이라는 텍스트를 추가
preds_txts_df.loc[preds_txts_df['label'] == 1, 'review_list'] = '[긍정] ' + preds_txts_df['review_list']

# 같은 행에서 df['label']의 값이 0이면 df['review_list']에 [부정]이라는 텍스트를 추가
preds_txts_df.loc[preds_txts_df['label'] == 0, 'review_list'] = '[부정] ' + preds_txts_df['review_list']

preds_txts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79853 entries, 0 to 79852
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   label        79853 non-null  int64  
 1   score        79853 non-null  float64
 2   review_list  79853 non-null  object 
dtypes: float64(1), int64(1), object(1)
memory usage: 1.8+ MB


In [66]:
preds_txts_df.reset_index(drop=True)

,label,score,review_list
0,1,0.978707,[긍정] 세안 혹은 샤워 후 얼굴 전체 적 으로 바르고 있습니다 자극 없이 순해서 좋어요
1,1,0.961900,[긍정] 한 달 사용 리뷰 입니다 추천 드립니다 좋은 상품 입니다
2,1,0.988946,[긍정] 매달 살 정도 합리 적 인 가격 과 좋은 제품 만나게 해주셔서 감사하고 꾸...
3,1,0.900858,[긍정] 눈가 주름 생기는거 같아 아이크 링 유명한 걸 n 또세 일하길래 구매 했어...
4,1,0.547701,[긍정] 처음 써 보는 건데 아직 특별한 점 잘 못 느끼겠어요 좀 더 써 봐야 할 ...
...,...,...,...
79848,1,0.952779,[긍정] n 각질 에도 좋고 일단 순한 성분 진정 되는 느낌 좋아서 매일 쓰고 있습...
79849,1,0.984017,[긍정] 재구매 했어용 피부 편안한 느낌 적 인 느낌 ㅎㅎ
79850,1,0.964095,[긍정] 기초 케어 라인 전부 후시 다인 으로 통일 해서 쓰고 있는데 유수 분 밸런...
79851,1,0.978629,[긍정] 후시 다인 제품 n 사용 해보고 반해가지고 n 요즘 매 일 매일 사용 하다...


In [67]:
# 긍정 댓글의 수
preds_txts_df.loc[preds_txts_df['label'] ==1].count()

,0
label,66061
score,66061
review_list,66061


In [68]:
# 부정 댓글의 수
preds_txts_df.loc[preds_txts_df['label'] ==0].count()

,0
label,13792
score,13792
review_list,13792


In [69]:
preds_txts_df.to_csv('/content/drive/MyDrive/김원/Datas/total_review_labeling_with_classification.csv')